# **Dataset Setup**

In [3]:
import numpy as np
import pandas as pd
import os
import zipfile

# Path to your zip file
zip_path = "/content/archive.zip"
# Directory where you want to extract the contents
extract_path = "/content/"

# Unzip the file
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_path)
        print("Files extracted successfully!")

# Load Dataset
df = pd.read_csv("insurance.csv")

# One-Hot Encode Categorical Features
df = pd.get_dummies(
    df,
    columns=["sex", "smoker", "region"],
    drop_first=True
).astype(float)

# Features and Target
X = df.drop("charges", axis=1).values
y = df["charges"].values.reshape(-1, 1)

# Feature Scaling
X_mean, X_std = X.mean(axis=0), X.std(axis=0)
X = (X - X_mean) / X_std

# Target scaling
y_mean, y_std = y.mean(), y.std()
y = (y - y_mean) / y_std

# Train-Test Split
np.random.seed(42)
indices = np.random.permutation(len(X))
split = int(0.8 * len(X))

X_train, y_train = X[indices[:split]], y[indices[:split]]
X_test, y_test = X[indices[split:]], y[indices[split:]]

Files extracted successfully!


# **Perceptron Class Definition**

In [6]:
class PerceptronRegressor:
    def __init__(self, learning_rate=0.01, epochs=5000, batch_size=32, shuffle=True):
        self.lr = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.W = None
        self.b = 0.0

    def fit(self, X, y):
        n_samples, n_features = X.shape
        # Initialize weights
        self.W = np.random.randn(n_features, 1) * 0.01
        self.b = 0.0

        for epoch in range(self.epochs):
            # 1. Shuffle data at the start of each epoch to prevent ordering bias
            if self.shuffle:
                indices = np.random.permutation(n_samples)
                X_shuffled = X[indices]
                y_shuffled = y[indices]
            else:
                X_shuffled = X
                y_shuffled = y

            # 2. Loop through the dataset in mini-batches
            for i in range(0, n_samples, self.batch_size):
                X_batch = X_shuffled[i : i + self.batch_size]
                y_batch = y_shuffled[i : i + self.batch_size]

                # Dynamic batch size adjustment for the final leftover batch
                current_batch_size = X_batch.shape[0]

                # Forward pass for current batch
                y_pred_batch = np.dot(X_batch, self.W) + self.b

                # Gradients calculated purely on this batch
                dW = (2 / current_batch_size) * np.dot(X_batch.T, (y_pred_batch - y_batch))
                db = (2 / current_batch_size) * np.sum(y_pred_batch - y_batch)

                # Step/Update
                self.W -= self.lr * dW
                self.b -= self.lr * db

            # Print entire dataset loss every 500 epochs to track overall progress
            if epoch % 500 == 0:
                full_y_pred = np.dot(X, self.W) + self.b
                total_loss = np.mean((full_y_pred - y) ** 2)
                print(f"Epoch {epoch}, Total Dataset Loss = {total_loss:.6f}")

    def predict(self, X):
        return np.dot(X, self.W) + self.b

# **Training and Evaluation**

In [7]:
# Initialize and train model
model = PerceptronRegressor(learning_rate=0.01, epochs=10000, batch_size=16)
model.fit(X_train, y_train)

# Testing
y_pred_scaled = model.predict(X_test)

# Inverse transform scaled target values back to original dollars
y_pred = y_pred_scaled * y_std + y_mean
y_actual = y_test * y_std + y_mean

# Performance Metrics
mse = np.mean((y_pred - y_actual) ** 2)
rmse = np.sqrt(mse)
ss_res = np.sum((y_actual - y_pred) ** 2)
ss_tot = np.sum((y_actual - np.mean(y_actual)) ** 2)
r2 = 1 - (ss_res / ss_tot)

print("\n--- Results ---")
print(f"RMSE: {rmse:.2f}")
print(f"R² Score: {r2:.4f}")

Epoch 0, Total Dataset Loss = 0.296110
Epoch 500, Total Dataset Loss = 0.245882
Epoch 1000, Total Dataset Loss = 0.246055
Epoch 1500, Total Dataset Loss = 0.245659
Epoch 2000, Total Dataset Loss = 0.245615
Epoch 2500, Total Dataset Loss = 0.245902
Epoch 3000, Total Dataset Loss = 0.245682
Epoch 3500, Total Dataset Loss = 0.245824
Epoch 4000, Total Dataset Loss = 0.245565
Epoch 4500, Total Dataset Loss = 0.245460
Epoch 5000, Total Dataset Loss = 0.245650
Epoch 5500, Total Dataset Loss = 0.245545
Epoch 6000, Total Dataset Loss = 0.245881
Epoch 6500, Total Dataset Loss = 0.245636
Epoch 7000, Total Dataset Loss = 0.245715
Epoch 7500, Total Dataset Loss = 0.245629
Epoch 8000, Total Dataset Loss = 0.245687
Epoch 8500, Total Dataset Loss = 0.245800
Epoch 9000, Total Dataset Loss = 0.245551
Epoch 9500, Total Dataset Loss = 0.245551

--- Results ---
RMSE: 6232.73
R² Score: 0.7512


# **Single Prediction Setup**

In [8]:
# Predict New Person
new_person = {
    "age": 35,
    "bmi": 28.5,
    "children": 2,
    "sex_male": 1,
    "smoker_yes": 0,
    "region_northwest": 0,
    "region_southeast": 1,
    "region_southwest": 0
}

new_df = pd.DataFrame([new_person])
new_X = (new_df.values - X_mean) / X_std

# Predict using the class method
pred_scaled = model.predict(new_X)
pred_charge = pred_scaled * y_std + y_mean

print(f"Predicted Insurance Charge: ${pred_charge[0][0]:.2f}")

Predicted Insurance Charge: $6477.40
